# 02_03_Profiling and Cleaning `territorialdivisions_3812` Geodataset #

In [1]:
from pathlib import Path

import geopandas as gpd
import pyogrio

In [2]:
path_raw = Path("../data/raw/")
path_geolocalisation = Path("../data/raw/geolocalisation/")
path_intermediate = Path("../data/intermediate/")

# Table of Contents

- [3. TERRITORIAL DIVISIONS GEODATASET](#3-territorial-divisions-geodataset)
- [3.1. Data Profiling](#31-data-profiling)
    - [3.1.1. Overview](#311-overview)
- [3.2. Data Quality and Cleaning](#32-data-quality--cleaning)
    - [3.2.1. Converting Types](#321-converting-types)
    - [3.2.2. Dropping Irrelevant Columns](#322-dropping-irrelevant-columns)
    - [3.2.3. Renaming Columns](#323-renaming-columns)
- [3.3. Export to Silver Layer](#33-export-to-silver-layer)

## 3. TERRITORIAL DIVISIONS GEODATASET ##

This dataset will be used for two purposes:  
1) As a **background layer for map visualizations** in Power BI.  

2) To enable spatial joins between `stations_cleaned` and `municipalities_flattened` in order to build the `station` dimension.

## 3.1. Data Profiling ##

### 3.1.1. Overview ###

* The `territorialdivisions_3812.gpkg` geodataset contains 16 layers. For this analysis, we only use the **municipality** layer.  

* Values in the `niscode` column are unique. This column will be used as a **Foreign Key** for spatial joins, when we build the `station` dimension.

* Values in `tgid`, `geometry`, `namefre` and `namedut` columns are also unique.

* We keep `tgid` as a **Business Key**.

* The CRS is **EPSG:3812**. We keep this projection for building the `station` dimension, and then convert it to **EPSG:4326** to prepare Power BI map visualization.

* There are neither missing nor duplicated values in this geodataframe.

In [3]:
layers = pyogrio.list_layers(path_geolocalisation / "territorialdivisions_3812.gpkg")
print([layer[0] for layer in layers])

['municipalsectioncenter', 'municipalsection', 'municipalcenter', 'belgianmaritimezone', 'belgianterritory', 'bordermarker', 'region', 'province', 'arrondissement', 'municipality', 'statisticalsector', 'electoralcanton', 'peacecourt', 'judicialcanton', 'judicialarrondissement', 'postaldistrict']


In [4]:
territorial_divisions = gpd.read_file(
    path_geolocalisation / "territorialdivisions_3812.gpkg",
    layer="municipality"
    )
territorial_divisions.head()

,tgid,modifdate,arrondissementcapital,provincecapital,regioncapital,countrycapital,niscode,city,languagestatute,nameger,namefre,namedut,geometry
0,{8BF44CB0-B8FD-44F6-A64F-1307610DA4C9},2007-01-05,False,False,False,False,72004,1,1,Bree,Bree,Bree,"MULTIPOLYGON Z (((735277.943 700725.863 0, 735..."
1,{54A85359-4967-4318-AA63-D234DDED2FD7},2007-01-05,False,False,False,False,63004,0,2,Baelen,Baelen,Baelen,"MULTIPOLYGON Z (((761804.571 647035.961 0, 761..."
2,{95E2AAE2-F9DB-456C-A113-2A21ED6F932F},2007-01-05,False,False,False,False,13003,0,1,Balen,Balen,Balen,"MULTIPOLYGON Z (((708505.572 703629.811 0, 708..."
3,{4487B4B8-4422-4856-97C5-33174BF84028},2007-01-05,False,False,False,False,62011,0,2,Bassenge,Bassenge,Bitsingen,"MULTIPOLYGON Z (((736728.393 660624.972 0, 736..."
4,{68A224E9-B2E7-4225-9FDA-03AE2B9C8C41},2007-01-05,False,False,False,False,85046,0,2,Habay,Habay,Habay,"MULTIPOLYGON Z (((735041.579 544346.475 0, 734..."


In [5]:
territorial_divisions.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 565 entries, 0 to 564
Data columns (total 13 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   tgid                   565 non-null    str           
 1   modifdate              565 non-null    datetime64[ms]
 2   arrondissementcapital  565 non-null    bool          
 3   provincecapital        565 non-null    bool          
 4   regioncapital          565 non-null    bool          
 5   countrycapital         565 non-null    bool          
 6   niscode                565 non-null    str           
 7   city                   565 non-null    int32         
 8   languagestatute        565 non-null    int32         
 9   nameger                565 non-null    str           
 10  namefre                565 non-null    str           
 11  namedut                565 non-null    str           
 12  geometry               565 non-null    geometry      
dt

In [6]:
print(territorial_divisions.crs)
territorial_divisions.columns

EPSG:3812


Index(['tgid', 'modifdate', 'arrondissementcapital', 'provincecapital',
       'regioncapital', 'countrycapital', 'niscode', 'city', 'languagestatute',
       'nameger', 'namefre', 'namedut', 'geometry'],
      dtype='str')

In [7]:
territorial_divisions.nunique()

tgid                     565
modifdate                  5
arrondissementcapital      2
provincecapital            2
regioncapital              2
countrycapital             2
niscode                  565
city                       2
languagestatute            7
nameger                  565
namefre                  564
namedut                  565
geometry                 565
dtype: int64

In [8]:
territorial_divisions.isnull().sum()

tgid                     0
modifdate                0
arrondissementcapital    0
provincecapital          0
regioncapital            0
countrycapital           0
niscode                  0
city                     0
languagestatute          0
nameger                  0
namefre                  0
namedut                  0
geometry                 0
dtype: int64

In [9]:
print(territorial_divisions["tgid"].duplicated().sum())
print(territorial_divisions["niscode"].duplicated().sum())
print(territorial_divisions["geometry"].duplicated().sum())
territorial_divisions.duplicated().sum()

0
0
0


np.int64(0)

## 3.2. Data Quality & Cleaning

### 3.2.1. Converting Types

To prepare the `niscode` column as a future **Foreign Key** for spatial joins, we convert it from `str` to `int`.

In [10]:
territorial_divisions["niscode"] = territorial_divisions["niscode"].astype(int)

### 3.2.2. Dropping Irrelevant Columns

The columns related to capitals, the `modifdate` column and the German names of the Belgian municipalities are irrelevant for our analysis. We exclude them.

In [11]:
columns_to_drop = [
    'modifdate', 
    'arrondissementcapital', 
    'provincecapital',  
    'regioncapital', 
    'countrycapital', 
    'city', 
    'nameger'
    ]

In [12]:
territorial_divisions = territorial_divisions.drop(columns=columns_to_drop)

In [13]:
territorial_divisions.columns

Index(['tgid', 'niscode', 'languagestatute', 'namefre', 'namedut', 'geometry'], dtype='str')

### 3.2.3. Renaming Columns

* We rename `niscode` as `CD_REFNIS` to facilitate future spatial joins.

* For better clarity, we rename `namefre` and `namedut` as `municipality_name_fre` and `municipality_name_dut`.

In [14]:
territorial_divisions = territorial_divisions.rename(columns={
    "niscode" : "CD_REFNIS",
    "namefre" : "municipality_name_fre",
    "namedut" : "municipality_name_dut"
})

In [15]:
territorial_divisions.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 565 entries, 0 to 564
Data columns (total 6 columns):
 #   Column                 Non-Null Count  Dtype   
---  ------                 --------------  -----   
 0   tgid                   565 non-null    str     
 1   CD_REFNIS              565 non-null    int64   
 2   languagestatute        565 non-null    int32   
 3   municipality_name_fre  565 non-null    str     
 4   municipality_name_dut  565 non-null    str     
 5   geometry               565 non-null    geometry
dtypes: geometry(1), int32(1), int64(1), str(3)
memory usage: 55.1 KB


The geodataframe is now ready for export to the intermediate directory.

## 3.3. Export to Silver Layer

In [16]:
territorial_divisions.to_file(path_intermediate / "territorial_divisions_3812_prepared.gpkg")

In [17]:
gdf_to_verify = gpd.read_file(path_intermediate / "territorial_divisions_3812_prepared.gpkg")
print(gdf_to_verify.shape, gdf_to_verify.dtypes.to_dict())
gdf_to_verify.head()

(565, 6) {'tgid': <StringDtype(na_value=nan)>, 'CD_REFNIS': dtype('int64'), 'languagestatute': dtype('int32'), 'municipality_name_fre': <StringDtype(na_value=nan)>, 'municipality_name_dut': <StringDtype(na_value=nan)>, 'geometry': <geopandas.array.GeometryDtype object at 0x0000013CFAE27920>}


,tgid,CD_REFNIS,languagestatute,municipality_name_fre,municipality_name_dut,geometry
0,{8BF44CB0-B8FD-44F6-A64F-1307610DA4C9},72004,1,Bree,Bree,"MULTIPOLYGON Z (((735277.943 700725.863 0, 735..."
1,{54A85359-4967-4318-AA63-D234DDED2FD7},63004,2,Baelen,Baelen,"MULTIPOLYGON Z (((761804.571 647035.961 0, 761..."
2,{95E2AAE2-F9DB-456C-A113-2A21ED6F932F},13003,1,Balen,Balen,"MULTIPOLYGON Z (((708505.572 703629.811 0, 708..."
3,{4487B4B8-4422-4856-97C5-33174BF84028},62011,2,Bassenge,Bitsingen,"MULTIPOLYGON Z (((736728.393 660624.972 0, 736..."
4,{68A224E9-B2E7-4225-9FDA-03AE2B9C8C41},85046,2,Habay,Habay,"MULTIPOLYGON Z (((735041.579 544346.475 0, 734..."
